In [0]:
%sql
-- Preview your unified Gold Customer Dimension
SELECT * FROM client_analytics_th.dim_customers LIMIT 5;

In [0]:
%sql
-- Directly query and display the Gold Transactions table inside the notebook
SELECT * FROM client_analytics_th.fct_transactions LIMIT 5;

In [0]:
%sql
-- Preview your unified Silver Customer Dimension
SELECT * FROM client_analytics_th.silver_customers LIMIT 5;

In [0]:
%sql
-- Preview your unified Silver Transaction Fact
SELECT * FROM client_analytics_th.silver_transactions LIMIT 5;

In [0]:
# -------------------------------------------------------------------------
# HIGH-FIDELITY END-TO-END PIPELINE VERIFICATION INTERACTIVE PREVIEW
# -------------------------------------------------------------------------
import pyspark.sql.functions as F

# Define the target schema
DATABASE_PREFIX = "client_analytics_th"

# 1. Map out the lineage tables across all 3 layers of your architecture
tables_to_preview = [
    # --- BRONZE LAYER ---
    {"name": f"{DATABASE_PREFIX}.bronze_client_a_customers", "desc": "1a. Bronze: Client A Customers (Raw Ingestion Lineage)"},
    {"name": f"{DATABASE_PREFIX}.bronze_client_a_transactions", "desc": "1b. Bronze: Client A Transactions (Raw Ingestion Lineage)"},
    {"name": f"{DATABASE_PREFIX}.bronze_client_b_customers", "desc": "1c. Bronze: Client B Customers (Raw Ingestion Lineage)"},
    {"name": f"{DATABASE_PREFIX}.bronze_client_b_transactions", "desc": "1d. Bronze: Client B Transactions (Raw Ingestion Lineage)"},
    {"name": f"{DATABASE_PREFIX}.bronze_client_c_customers", "desc": "1e. Bronze: Client C Customers (Raw Ingestion Lineage)"},
    {"name": f"{DATABASE_PREFIX}.bronze_client_c_transactions", "desc": "1f. Bronze: Client C Transactions (Raw Ingestion Lineage)"},
    
    # # --- SILVER LAYER ---
    # {"name": f"{DATABASE_PREFIX}.silver_customers", "desc": "2a. Silver: Harmonized Customers (Unified Structs & Split Names)"},
    # {"name": f"{DATABASE_PREFIX}.silver_transactions", "desc": "2b. Silver: Harmonized Transactions (try_cast & Character Cleans Applied)"},
    
    # # --- GOLD LAYER ---
    # {"name": f"{DATABASE_PREFIX}.dim_customers", "desc": "3a. Gold: Customer Dimension Table (Unique Hashed grain)"},
    # {"name": f"{DATABASE_PREFIX}.fct_transactions", "desc": "3b. Gold: Transaction Fact Table (Universal Foreign Keys)"}
]

# 2. Iterate and render independent interactive UI grids for each table
for table in tables_to_preview:
    # Print a clear, formatted text heading right above each interactive grid
    print(f"\n" + "="*80)
    print(f" 📂 {table['desc']}")
    print("="*80)
    
    try:
        # Load exactly 5 rows from the Delta log
        sample_df = spark.read.table(table["name"]).limit(5)
        
        # Trigger the native interactive Databricks HTML sheet component
        display(sample_df)
    except Exception as e:
        print(f"❌ Failed to load preview grid for {table['name']}. Details: {str(e)}")

# 3. Render the Final Verification Join (Proves Surrogate Keys connect perfectly)
print(f"\n" + "="*80)
print(f" 🏆 4. GOLD VERIFICATION JOIN (Validates customer_sk Integration Key Bridges)")
print("="*80)

try:
    fct_df = spark.read.table(f"{DATABASE_PREFIX}.fct_transactions")
    dim_df = spark.read.table(f"{DATABASE_PREFIX}.dim_customers")
    
    join_preview = fct_df.alias("f").join(
        dim_df.alias("c"),
        F.col("f.customer_sk") == F.col("c.customer_sk"),
        "inner"
    ).select(
        F.col("f.transaction_sk"),
        F.col("c.source_client"),
        F.col("c.first_name"),
        F.col("c.last_name"),
        F.col("f.amount"),
        F.col("f.transaction_timestamp")
    ).limit(5)
    
    display(join_preview)
except Exception as e:
    print(f"❌ Verification Join failed execution. Details: {str(e)}")